# Qwen3 replication + OPEN-1B pilot
Run All creates controls only. Read README.md first. Run one GPU experiment at a time. Qwen and native OPEN use separate Python environments.

In [ ]:
from pathlib import Path
import sys, json, subprocess
import controls, paired_suite, open_pilot, environment_setup

OLMO_SOURCE = Path('/home/ubuntu/1/runs/olmo_association_v1')
PREVIOUS = Path('/home/ubuntu/4/runs/paired_base_instruct_v1')  # or your paired_models_share.zip
OUTPUT_QWEN = Path.cwd() / 'runs/qwen3_natural_v1'
OUTPUT_OPEN = Path.cwd() / 'runs/open1b_history_pilot_v1'
QWEN_ENV = Path.cwd() / 'env_qwen3'
QWEN_PYTHON = QWEN_ENV / 'bin/python'
OPEN_PYTHON = Path('/home/ubuntu/open1b_audit_env/bin/python')

QWEN = paired_suite.defaults(OLMO_SOURCE, PREVIOUS, OUTPUT_QWEN)
OPEN = open_pilot.defaults(OUTPUT_OPEN)
# Optional before FIRST launch:
# QWEN['path_seeds'] = [1, 2, 3]
# OPEN['probe_token_ids'] = [...]  # fixed diagnostic/retention token IDs, same tokenizer
# OPEN['steps'] = 1              # actual original global-batch pretraining steps

OTHER_GPU_RUNS = [Path('/home/ubuntu/4/runs/frame_history_v1'),
                  Path('/home/ubuntu/4/runs/paired_base_instruct_v1'),
                  Path('/home/ubuntu/4/runs/multimodel_paper_v1')]

def guard(target):
    for p in OTHER_GPU_RUNS + [OUTPUT_QWEN, OUTPUT_OPEN]:
        if p.resolve() != Path(target).resolve() and controls.alive(p):
            raise RuntimeError(f'Another listed study is running: {p}. Stop/wait before launching.')

def setup_qwen():
    global QWEN_PYTHON
    QWEN_PYTHON = Path(environment_setup.setup_qwen(QWEN_ENV))

def launch_qwen():
    guard(OUTPUT_QWEN)
    controls.start_qwen(QWEN, QWEN_PYTHON)

def refresh_qwen(): return controls.status(OUTPUT_QWEN)
def stop_qwen(): controls.stop(OUTPUT_QWEN)
def export_qwen():
    if controls.alive(OUTPUT_QWEN): raise RuntimeError('Stop Qwen and wait for alive=False before export.')
    command = 'import paired_suite,sys;print(paired_suite._export(__import__("pathlib").Path(sys.argv[1])))'
    r = subprocess.run([str(QWEN_PYTHON), '-c', command, str(OUTPUT_QWEN)], text=True, capture_output=True)
    print(r.stdout or r.stderr)
    if r.returncode: raise RuntimeError('Export failed; see message above.')

def check_open():
    if not OPEN_PYTHON.exists():
        print('Native environment not found. Set OPEN_PYTHON to an official compatible Gensyn replay environment. See README, especially GH200/ARM64 support.')
        return
    environment_setup.check_native(OPEN_PYTHON)

def launch_open():
    guard(OUTPUT_OPEN)
    if not OPEN_PYTHON.exists(): raise FileNotFoundError('Set OPEN_PYTHON to the compatible native replay environment first.')
    controls.start_pilot(OPEN, OPEN_PYTHON, 'native')

def audit_open_proxies():
    guard(OUTPUT_OPEN)
    controls.start_pilot(OPEN, QWEN_PYTHON, 'proxy')

def refresh_open(): return controls.status(OUTPUT_OPEN)
def stop_open(): controls.stop(OUTPUT_OPEN)
def export_open(): return controls.export(OUTPUT_OPEN)
def show_reports():
    for root in [OUTPUT_QWEN, OUTPUT_OPEN]:
        print('\n', root)
        for name in ['REPORT.txt', 'pilot_gate.json', 'replay_result.json']:
            p=root/name
            if p.exists(): print(p.read_text()[:12000])


## Controls
Qwen: Setup environment once → Launch → Refresh.

OPEN: Check readiness → Native pilot → inspect replay and derivative gates → Proxy audits. A dependency block is not a scientific negative result. There is no automatic fallback to fresh momentum or to HF inference gradients.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    panel=widgets.Output()
    def button(label, function):
        b=widgets.Button(description=label, layout=widgets.Layout(width='180px'))
        def click(_):
            with panel:
                panel.clear_output(wait=True)
                try: function()
                except Exception as exc: print(type(exc).__name__ + ': ' + str(exc))
        b.on_click(click)
        return b
    display(widgets.HBox([button('Setup Qwen env',setup_qwen),button('Launch Qwen',launch_qwen),button('Refresh Qwen',refresh_qwen),button('Stop Qwen',stop_qwen)]))
    display(widgets.HBox([button('Check OPEN readiness',check_open),button('Native OPEN pilot',launch_open),button('OPEN proxy audits',audit_open_proxies)]))
    display(widgets.HBox([button('Refresh OPEN',refresh_open),button('Stop OPEN',stop_open),button('Export OPEN',export_open),button('Export Qwen',export_qwen)]))
    display(button('Show reports',show_reports),panel)
except ImportError:
    print('Widgets unavailable. Call the functions above in separate cells, e.g. launch_qwen() and refresh_qwen().')
